In [9]:
import numpy as np
import pandas as pd
import os

In [10]:
!git clone https://github.com/jenilrupareliya5150-bit/FlyRankAi-ml-Track.git

Cloning into 'FlyRankAi-ml-Track'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 193 (delta 88), reused 89 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 2.67 MiB | 4.03 MiB/s, done.
Resolving deltas: 100% (88/88), done.


In [11]:
%cd FlyRankAi-ml-Track

/content/FlyRankAi-ml-Track/FlyRankAi-ml-Track


In [17]:
df=pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Freshness signals

The paper reports that freshness is associated with a better Health Score, especially for longer content, and suggests that freshness can amplify editing quality rather than replace it.

**Methodology question:** How exactly was freshness measured, and was the Health Score measured using an independent outcome that was not derived from the same freshness-related features?

### Finding 2: Feature importance

The paper reports that some features appear more important than others for explaining or predicting the outcome.

**Methodology question:** Was feature importance evaluated on held-out data, and were correlated or potentially leaky features checked before interpreting the importance results?

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest grouped split

In Week 5, the model was evaluated on a train/test split. For this audit, I re-ran the model using a grouped split by `client_id` so that a client appears entirely in either the training set or the test set, but not both. This gives a stricter test of whether the model can generalize to clients it has not seen during training.

I compare the Precision@50 before and after this change. A decrease after grouping would indicate that some of the previous performance may have depended on seeing the same clients during training and testing.

In [19]:
# Create the target used for the model
df["declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["declining_label"].value_counts())

declining_label
1    16262
0    13738
Name: count, dtype: int64


In [20]:
from sklearn.model_selection import GroupShuffleSplit

# Target column
target = "declining_label"

# Create a client-grouped split
# This makes sure the same client does not appear in both train and test
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        df[target],
        groups=df["client_id"]
    )
)

# Create train and test data
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

# Check client overlap
client_overlap = (
    set(train_df["client_id"])
    & set(test_df["client_id"])
)

print("\nClient overlap:", len(client_overlap))

Training rows: 23837
Testing rows: 6163

Training clients: 25
Testing clients: 7

Client overlap: 0


In [21]:
print("Target column:", target)
print("\nTarget distribution:")
print(df[target].value_counts())

print("\nTarget percentages:")
print(df[target].value_counts(normalize=True) * 100)

Target column: declining_label

Target distribution:
declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
declining_label
1    54.206667
0    45.793333
Name: proportion, dtype: float64


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Target
target = "declining_label"

# Columns that must NOT be used as features
drop_cols = [
    "declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

# Create feature list
feature_cols = [
    col for col in df.columns
    if col not in drop_cols
]

# Create X and y using the honest split
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df[target]
y_test = test_df[target]

print("Number of features:", len(feature_cols))
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Number of features: 40
Training shape: (23837, 40)
Testing shape: (6163, 40)


In [23]:
# Separate numerical and categorical features

numeric_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))

Numerical features: 29
Categorical features: 11


In [24]:
# Numerical preprocessing
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ]
)

# Complete model
honest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# Train
honest_model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [25]:
# Predict probabilities for the positive class
honest_probability = honest_model.predict_proba(X_test)[:, 1]

# Calculate ROC-AUC
honest_auc = roc_auc_score(
    y_test,
    honest_probability
)

print("Honest-split ROC-AUC:", honest_auc)

Honest-split ROC-AUC: 0.8378458481990365


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### 3. Leakage audit

I checked the final feature set for possible target leakage. The target `declining_label` is derived from `trend_direction`, so `trend_direction` should not be used as a model feature. I also excluded `trend_pct` because it is directly related to the trend information used to construct the label.

The leakage check showed that these known leakage columns were not included in the final feature set. I also checked the identifier columns `content_id` and `client_id`; neither is being used as a model feature. `client_id` is used only for grouped train/test splitting.

Therefore, the final feature set does not include the identified target-derived or identifier columns. This reduces the risk that the model is using information that would not be legitimately available for prediction.

In [26]:
# Section 3: Leakage Audit

target = "declining_label"

# Columns that should not be used as model features
leakage_cols = [
    "declining_label",
    "trend_direction",
    "trend_pct"
]

print("Target:", target)

print("\nColumns checked for leakage:")
for col in leakage_cols:
    print("-", col)

Target: declining_label

Columns checked for leakage:
- declining_label
- trend_direction
- trend_pct


In [27]:
print("Target creation:")
print("declining_label = 1 when trend_direction == 'down'")

print("\nTrend direction vs target:")
print(
    pd.crosstab(
        df["trend_direction"],
        df["declining_label"]
    )
)

Target creation:
declining_label = 1 when trend_direction == 'down'

Trend direction vs target:
declining_label     0      1
trend_direction             
down                0  16262
flat             1152      0
new              2236      0
stable           5962      0
up               4388      0


In [28]:
print("Trend percentage statistics:")
print(df["trend_pct"].describe())

print("\nAverage trend_pct by target:")
print(
    df.groupby("declining_label")["trend_pct"].mean()
)

Trend percentage statistics:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Average trend_pct by target:
declining_label
0    79.003179
1   -58.113830
Name: trend_pct, dtype: float64


In [29]:
print("Current model features:")
print(feature_cols)

print("\nLeakage columns accidentally included:")
print(
    [col for col in leakage_cols if col in feature_cols]
)

Current model features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Leakage columns accidentally included:
[]


In [30]:
id_cols = [
    "content_id",
    "client_id"
]

print("ID columns:")
for col in id_cols:
    print("-", col)

print("\nIDs used as model features:")
print(
    [col for col in id_cols if col in feature_cols]
)

ID columns:
- content_id
- client_id

IDs used as model features:
[]


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### 4. Claim rewrite

The model results show that Logistic Regression achieved a higher Precision@50 than the Week-5 baseline on the evaluated test set. This indicates that the model may provide a useful ranking signal for identifying content associated with the declining label.

However, this result does not prove that the model will identify all declining webpages or that the model will perform the same way on future data. The result should therefore be treated as decision-support evidence rather than a guarantee.

A safer claim is: the model provides a measured ranking signal that can help prioritize webpages for further review and possible refresh.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.